In [21]:
import numpy as np
from numpy.linalg import matrix_rank
from scipy.linalg import svd, diagsvd, inv, pinv

In [22]:
A = np.array([  # tall/thin, full column rank
    [2, 0],
    [1, -2], 
    [3, 4]])
A

array([[ 2,  0],
       [ 1, -2],
       [ 3,  4]])

In [23]:
M, N = A.shape
I_M, I_N = np.eye(M), np.eye(N)
R = matrix_rank(A)
R, R == np.min((M, N))  # rank 2 = full column rank

(np.int64(2), np.True_)

In [24]:
# pure column space (CS)
y1 = np.array([[2], [-1], [7]])
y1

array([[ 2],
       [-1],
       [ 7]])

In [25]:
# colum space + left null space (LNS)
y2 = np.array([[-1/2], [1], [8]])
y2

array([[-0.5],
       [ 1. ],
       [ 8. ]])

In [26]:
# pure left null space
e_LNS = y2 - y1
e_LNS, A.T @ e_LNS

(array([[-2.5],
        [ 2. ],
        [ 1. ]]),
 array([[0.],
        [0.]]))

In [27]:
A_li = pinv(A)  # for full column rank A, this yields the left inverse
A_li

array([[ 0.22222222,  0.22222222,  0.11111111],
       [-0.11111111, -0.21111111,  0.14444444]])

In [28]:
# make sure to understand difference between inv() and pinv()

In [29]:
inv(A.T @ A) @ A.T  # explicit equation for the left inverse

array([[ 0.22222222,  0.22222222,  0.11111111],
       [-0.11111111, -0.21111111,  0.14444444]])

In [30]:
# for y1 and y2 -> same solution xh, due to min ||e_LNS||_2 = ||y - Ax||_2 w.r.t. x
A_li @ y1, A_li @ y2

(array([[1.],
        [1.]]),
 array([[1.],
        [1.]]))

In [31]:
# projection matrices
P_CS = A @ A_li  # to column space
P_LNS = I_M - P_CS  # to left null space
P_RS = A_li @ A  # to row space (RS) -> identity matrix for tall/thin, full column rank A
P_NS = I_N - P_RS  # to null space (NS) -> zero matrix for tall/thin, full column rank A
P_CS, P_LNS, P_RS, P_NS

(array([[ 0.44444444,  0.44444444,  0.22222222],
        [ 0.44444444,  0.64444444, -0.17777778],
        [ 0.22222222, -0.17777778,  0.91111111]]),
 array([[ 0.55555556, -0.44444444, -0.22222222],
        [-0.44444444,  0.35555556,  0.17777778],
        [-0.22222222,  0.17777778,  0.08888889]]),
 array([[ 1.00000000e+00, -1.66533454e-16],
        [-1.66533454e-16,  1.00000000e+00]]),
 array([[2.22044605e-16, 1.66533454e-16],
        [1.66533454e-16, 0.00000000e+00]]))

In [32]:
# check projections
P_CS @ y1, P_CS @ y2, P_LNS @ y1, P_LNS @ y2

(array([[ 2.],
        [-1.],
        [ 7.]]),
 array([[ 2.],
        [-1.],
        [ 7.]]),
 array([[4.44089210e-16],
        [0.00000000e+00],
        [9.99200722e-16]]),
 array([[-2.5],
        [ 2. ],
        [ 1. ]]))

In [33]:
np.allclose(np.array([[0], [0], [0]]), P_LNS @ y1), np.allclose(e_LNS, P_LNS @ y2)

(True, True)

In [34]:
# pseudo-inverse via SVD
# special case: tall/thin, full column rank == left inverse
[U, s, Vt] = svd(A)
V = Vt.T
S = diagsvd(s, M, N)
A_li_svd = V @ diagsvd(1/s, N, M) @ U.T  # general concept of the pseudo inverse
A_li_svd, np.allclose(A_li, A_li_svd)

(array([[ 0.22222222,  0.22222222,  0.11111111],
        [-0.11111111, -0.21111111,  0.14444444]]),
 True)

In [35]:
V @ (inv(S.T @ S) @ S.T) @ U.T  # formulation for the left inverse using the S matrix

array([[ 0.22222222,  0.22222222,  0.11111111],
       [-0.11111111, -0.21111111,  0.14444444]])

In [36]:
# general formulation for the projections matrices based on SVD matrices
U[:,:R] @ U[:,:R].T  # P_CS

array([[ 0.44444444,  0.44444444,  0.22222222],
       [ 0.44444444,  0.64444444, -0.17777778],
       [ 0.22222222, -0.17777778,  0.91111111]])

In [37]:
U[:,R:] @ U[:,R:].T  # P_LNS

array([[ 0.55555556, -0.44444444, -0.22222222],
       [-0.44444444,  0.35555556,  0.17777778],
       [-0.22222222,  0.17777778,  0.08888889]])

In [38]:
V[:, :R] @ V[:, :R].T  # P_RS

array([[1., 0.],
       [0., 1.]])

In [39]:
V[:, R:] @ V[:, R:].T  # P_NS

array([[0., 0.],
       [0., 0.]])